# After-visit summary per encounter

Starting point for the cost workflow: extract the patient-facing **after-visit summary (AVS)** for each of the 25 encounters into one clean CSV.

The AVS is the natural entry point because it states, in plain language, what was discussed and the concrete next steps (medications to start, follow-ups, referrals, counseling). Later steps can parse these next steps into billable items and price them.

Output: `processed_csv/patient_to_after_visit_summary.csv` — one row per patient/encounter.

> Note: the AVS is `deterministic_extractive_v1` from the note's Assessment & Plan and is **not clinically reviewed**.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 100)

DATASET_NAME = "synthetic-ambient-fhir-25.jsonl"
DATASET_CANDIDATES = [
    Path("data/synthetic-ambient-fhir-25") / DATASET_NAME,
    Path("../data/synthetic-ambient-fhir-25") / DATASET_NAME,
]

DATASET_PATH = next((path for path in DATASET_CANDIDATES if path.exists()), None)
if DATASET_PATH is None:
    checked = "\n".join(f"- {path.resolve()}" for path in DATASET_CANDIDATES)
    raise FileNotFoundError(f"Dataset not found. Checked:\n{checked}")

with DATASET_PATH.open(encoding="utf-8") as handle:
    records = [json.loads(line) for line in handle if line.strip()]

assert len(records) == 25, f"Expected 25 encounters, found {len(records)}"
print(f"Loaded {len(records)} encounters from {DATASET_PATH.resolve()}")

In [ ]:
def patient_name(record):
    names = record["patient_context"]["patient"].get("name", [])
    if not names:
        return "Unknown"
    name = names[0]
    return " ".join([*name.get("given", []), name.get("family", "")]).strip()


def clean_line(line):
    """Strip bullet markers / leading punctuation and surrounding whitespace."""
    return re.sub(r"^[\s\u2022\u2023\u25e6\-\*•]+", "", line).strip()


# AVS headers we expect, in order. Everything between one header and the next
# belongs to that section; the leading "Visit summary" title is dropped.
SECTION_HEADERS = {
    "what_we_discussed": ["what we discussed"],
    "next_steps": ["next steps", "your next steps", "plan", "what happens next"],
}


def parse_avs_sections(avs_text):
    """Split an after-visit summary into ordered {section: [items]}."""
    header_lookup = {
        alias: key for key, aliases in SECTION_HEADERS.items() for alias in aliases
    }
    sections = {key: [] for key in SECTION_HEADERS}
    current = None
    for raw_line in avs_text.splitlines():
        line = clean_line(raw_line)
        if not line:
            continue
        normalized = line.lower().rstrip(":")
        if normalized in header_lookup:
            current = header_lookup[normalized]
            continue
        if normalized == "visit summary":
            current = None
            continue
        if current is not None:
            sections[current].append(line)
    return sections


rows = []
for patient_number, record in enumerate(records):
    metadata = record["metadata"]
    patient = record["patient_context"]["patient"]
    encounter = record["encounter_fhir"]["encounter"]
    avs_text = record.get("after_visit_summary", "") or ""
    sections = parse_avs_sections(avs_text)

    rows.append({
        "patient_number": patient_number,
        "patient_id": metadata["patient_id"],
        "encounter_id": metadata["encounter_id"],
        "patient": patient_name(record),
        "gender": patient.get("gender"),
        "birth_date": patient.get("birthDate"),
        "encounter_date": metadata["date"][:10],
        "visit_title": metadata.get("visit_title"),
        "facility": encounter.get("serviceProvider", {}).get("display"),
        "what_we_discussed": " | ".join(sections["what_we_discussed"]),
        "next_steps": " | ".join(sections["next_steps"]),
        "num_discussed": len(sections["what_we_discussed"]),
        "num_next_steps": len(sections["next_steps"]),
        "avs_word_count": len(avs_text.split()),
        "after_visit_summary": avs_text,
    })

avs_df = pd.DataFrame(rows)
print(f"Extracted after-visit summaries for {len(avs_df)} encounters")
display(
    avs_df[[
        "patient_number",
        "patient",
        "encounter_date",
        "visit_title",
        "num_discussed",
        "num_next_steps",
        "avs_word_count",
    ]]
)

## Inspect one after-visit summary

Change `PATIENT_NUMBER` (0–24) to read the full AVS and its parsed sections.

In [ ]:
PATIENT_NUMBER = 0  # choose 0 through 24

selected = avs_df.iloc[PATIENT_NUMBER]
display(selected[["patient", "encounter_date", "visit_title", "facility"]].to_frame())

display(Markdown("### What we discussed"))
display(pd.DataFrame({"item": selected["what_we_discussed"].split(" | ") if selected["what_we_discussed"] else []}))

display(Markdown("### Next steps"))
display(pd.DataFrame({"item": selected["next_steps"].split(" | ") if selected["next_steps"] else []}))

display(Markdown("### Raw after-visit summary"))
display(Markdown(selected["after_visit_summary"]))

## Write processed CSV

In [ ]:
processing_dir = Path("data_processing") if Path("data_processing").is_dir() else Path(".")
output_dir = processing_dir / "processed_csv"
output_dir.mkdir(parents=True, exist_ok=True)

avs_path = output_dir / "patient_to_after_visit_summary.csv"
avs_df.to_csv(avs_path, index=False)
print(f"Wrote {len(avs_df)} rows to {avs_path.resolve()}")

# Sanity checks
missing = avs_df[avs_df["avs_word_count"] == 0]
if len(missing):
    print(f"WARNING: {len(missing)} encounters have an empty after-visit summary")
no_next_steps = avs_df[avs_df["num_next_steps"] == 0]
if len(no_next_steps):
    print(f"NOTE: {len(no_next_steps)} encounters had no parseable 'Next steps' section:")
    display(no_next_steps[["patient_number", "patient", "visit_title"]])

## Full per-patient encounter export

One row per patient with the core encounter context plus the three free-text artifacts (transcript, clinical note, after-visit summary).

Output: `processed_csv/patient_encounter_details.csv`

In [ ]:
from datetime import date


def age_at(birth_date, encounter_date):
    """Whole years between birth date and encounter date (both YYYY-MM-DD)."""
    try:
        birth = date.fromisoformat(birth_date[:10])
        visit = date.fromisoformat(encounter_date[:10])
    except (TypeError, ValueError):
        return None
    return visit.year - birth.year - ((visit.month, visit.day) < (birth.month, birth.day))


def reason_for_visit(record):
    """Prefer the coded encounter reason; fall back to the human-readable visit title."""
    reasons = []
    for concept in record["encounter_fhir"]["encounter"].get("reasonCode", []):
        text = concept.get("text") or next(
            (c.get("display") for c in concept.get("coding", []) if c.get("display")), None
        )
        if text:
            reasons.append(text)
    if reasons:
        return "; ".join(dict.fromkeys(reasons))
    return record["metadata"].get("visit_title")


detail_rows = []
for patient_number, record in enumerate(records):
    metadata = record["metadata"]
    patient = record["patient_context"]["patient"]
    encounter_date = metadata["date"][:10]
    detail_rows.append({
        "patient_number": patient_number,
        "identifier": record["id"],
        "patient": patient_name(record),
        "reason_for_visit": reason_for_visit(record),
        "date": encounter_date,
        "sex": patient.get("gender"),
        "age": age_at(patient.get("birthDate", ""), encounter_date),
        "transcript": record.get("transcript", ""),
        "clinical_note": record.get("note", ""),
        "after_visit_summary": record.get("after_visit_summary", ""),
    })

details_df = pd.DataFrame(detail_rows)

details_path = output_dir / "patient_encounter_details.csv"
details_df.to_csv(details_path, index=False)
print(f"Wrote {len(details_df)} rows to {details_path.resolve()}")

display(
    details_df[[
        "patient_number",
        "identifier",
        "patient",
        "reason_for_visit",
        "date",
        "sex",
        "age",
    ]]
)